In [61]:
using LowLevelFEM, LinearAlgebra

In [62]:
openGeometry("rectangles.geo")

In [63]:
#openPreProcessor()

In [64]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u);

In [65]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0)
bc_top = BoundaryCondition("top", ux=0, uy=(x,y,z)->x*(x-10)/130)

K = ∫(SymGrad(U) ⋅ D(:PlaneStress, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

#showDoFResults(u, name="u", factor=1, visible=true)

nodal VectorField
[0.0; 0.0; … ; -0.026961545388719164; -0.17253959505900868;;]

In [66]:
C = contact(u, master="master", slave="slave", cn=1e6)

Contact("slave" -> "master", 101 candidate nodes, 67 active, stick=67, slip=0, G=(202, 1392), C=(202, 202))

In [67]:
support = [bc_bottom, bc_top]
free = freeDoFs(U, support)

u_it = copy(u)

old_tags = copy(C.master_element_tags)
old_G = copy(C.G)

for iter in 1:30

    updateContact!(C, u_it)

    nchanged = count(old_tags .!= C.master_element_tags)

    dG = norm(C.G - old_G) /
        max(norm(old_G), eps())

    println(
        "master changes = ", nchanged,
        ", dG = ", dG
    )

    old_tags = copy(C.master_element_tags)
    old_G = copy(C.G)

    gc = zeros(size(C.G, 1))

    for i in eachindex(C.slave_nodes)
        if C.active[i]
            gc[2i - 1] = C.gap_values[i]
        end
    end

    # Contact residual and tangent
    rc = C.G' * (C.C * gc)
    Kc = C.G' * C.C * C.G

    # Total residual
    r = K.A * u_it.a[:,1] - f.a[:,1] + rc

    # Current tangent
    A = K.A + Kc

    # Homogeneous correction on prescribed DoFs
    Δu = zeros(length(r))
    Δu[free] = -A[free, free] \ r[free]

    r0 = norm(r[free])

    α = 1.0
    u_trial = copy(u_it)

    while α > 1e-6
        u_trial.a[:, 1] .= u_it.a[:, 1] .+ α .* Δu

        updateContact!(C, u_trial)

        gc_trial = zeros(size(C.G, 1))
        for i in eachindex(C.slave_nodes)
            if C.active[i]
                gc_trial[2i - 1] = C.gap_values[i]
            end
        end

        rc_trial = C.G' * (C.C * gc_trial)

        r_trial =
            K.A * u_trial.a[:, 1] -
            f.a[:, 1] +
            rc_trial

        if norm(r_trial[free]) < r0
            break
        end

        α *= 0.5
    end

    u_it.a[:, 1] .= u_trial.a[:, 1]

    err = α * norm(Δu[free]) /
          max(norm(u_it.a[free,1]), eps())

    println(
        "iter = ", iter,
        ", α = ", α,
        ", active = ", count(C.active),
        ", min gap = ", minimum(C.gap_values),
        ", error = ", err,
        ", |r| = ", norm(r[free])
    )

    err < 1e-8 && break
end

u = u_it

master changes = 0, dG = 0.0
iter = 1, α = 1.0, active = 63, min gap = -0.00046808468817496944, error = 0.24439270921006415, |r| = 686226.0869707131
master changes = 0, dG = 0.011493771677437568
iter = 2, α = 1.0, active = 61, min gap = -0.00046812929093552764, error = 0.006837508759859004, |r| = 611.1734394907104
master changes = 0, dG = 0.007862389916560289
iter = 3, α = 1.0, active = 59, min gap = -0.00046798647500812164, error = 0.0015446530527191817, |r| = 239.41691423941782
master changes = 0, dG = 0.0014660499632039642
iter = 4, α = 1.0, active = 59, min gap = -0.0004679973558697437, error = 0.0004078771654333726, |r| = 62.713574816948
master changes = 0, dG = 0.00038063850564330146
iter = 5, α = 1.0, active = 59, min gap = -0.00046799703012708865, error = 9.136027742140322e-7, |r| = 0.0298144699075731
master changes = 0, dG = 2.0484437818209795e-6
iter = 6, α = 1.0, active = 59, min gap = -0.0004679970454692522, error = 9.745318357227852e-9, |r| = 0.0004888711928470854


nodal VectorField
[-0.0015923490245112616; -0.0004251619264149414; … ; -0.02584067820345526; -0.14034053617440198;;]

In [68]:
showDoFResults(u, name="u", factor=1, visible=true)

0

In [69]:
C.gap

elementwise ScalarField
[[0.07821289157419901; 0.07462436604440892;;], [0.07462436604440892; 0.07097445763459473;;], [0.07097445763459473; 0.06727566542842531;;], [0.06727566542842531; 0.06335443228532321;;], [0.06335443228532321; 0.059177453999172326;;], [0.059177453999172326; 0.054858446939348535;;], [0.054858446939348535; 0.0503400537840373;;], [0.0503400537840373; 0.045765954650683324;;], [0.045765954650683324; 0.041081451907531256;;], [0.041081451907531256; 0.036411364005641635;;]  …  [0.0362574787807798; 0.04092941998784293;;], [0.04092941998784293; 0.045581139605447706;;], [0.045581139605447706; 0.0502236360983142;;], [0.0502236360983142; 0.0547179517093705;;], [0.0547179517093705; 0.059112083048363684;;], [0.059112083048363684; 0.06330633873928575;;], [0.06330633873928575; 0.06716778503374547;;], [0.06716778503374547; 0.07085422273821125;;], [0.07085422273821125; 0.07450131479128717;;], [0.07450131479128717; 0.07805916550642589;;]]

In [70]:
openPostProcessor()